# DermaLens AI · EfficientNet-B0 · Real-only progressive fine-tuning · SMOKE / FULL

**Purpose:** Train an authentic ImageNet-pretrained **EfficientNet-B0**, our sixth and lightweight baseline, on the finalized **six-class real facial dermatology dataset**. This is **B0, not the older EfficientNet-B3**.

**Dataset:** [Facial Dermatology Image Dataset](https://www.kaggle.com/datasets/shanmukhadattaboda/facial-dermatology-image-dataset/data) (fixed `train/`, `validation/`, `test/`). Attach this Kaggle input; *do not* use the old unsplit real-image collection.

### One-switch execution

- `RUN_MODE = "SMOKE"` (default): balanced small subset per split; **one epoch per stage**, including the optional whole-backbone code path; verifies dataset mounting, authentic pretrained model load, backward passes, checkpoint reload, evaluation, standalone inference and ZIP export. Smoke metrics are **not research results**.
- `RUN_MODE = "FULL"`: all real images; 4 + 8 + 8 epochs (maximum before per-stage early stopping). The optional complete-backbone stage is off by default.
- Run via **Kaggle Save Version → Save & Run All** with **T4 GPU** and Internet enabled to download the official weights on a fresh session. No manual prompts or package upgrades.

### Progressive training design

1. **Head:** classifier only.
2. **Final MBConv stage:** classifier + `features.7` and final 1×1 projection `features.8`.
3. **Final two MBConv stages:** classifier + `features.6`, `features.7`, `features.8`.
4. **Optional full model:** all features with small discriminative learning rates. Tested in SMOKE, disabled by default in FULL.

Every stage reloads its best **validation macro-F1** checkpoint before the next. Best overall checkpoint is selected without looking at test data. Train-only mild augmentation and aspect-ratio-preserving 224×224 padding are recorded for downstream inference. The original ImageNet recipe uses resize/center-crop, but we intentionally preserve the entire clinical face here; this design choice must be reported when comparing architectures.

**Verification limits:** a SMOKE pass tests the pipeline but cannot guarantee uninterrupted FULL training or medically valid labels. A saved model is not a clinical diagnostic tool. The notebook saves reproducibility metadata, class mappings, all stage and final checkpoints, logs, CSV/JSON metrics, plots, a standalone CPU/GPU inference script, and a ZIP bundle.

## Cell 1 — Imports, package versions, device and reproducibility

Use built-in Kaggle PyTorch/TorchVision, Pillow, scikit-learn, numpy, pandas and matplotlib. **No `pip install` or notebook runtime restart** is needed for EfficientNet-B0. We call the current TorchVision `weights=` interface instead of deprecated `pretrained=True`. Print installed versions and detected GPU so full-run provenance is saved later.

In [ ]:
import os, sys, json, csv, time, random, hashlib, logging, traceback, shutil, zipfile, math
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime, timezone
from contextlib import nullcontext

import numpy as np
import pandas as pd
from PIL import Image, ImageOps, ImageFile, UnidentifiedImageError
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from torchvision.transforms import InterpolationMode
from torchvision.models import EfficientNet_B0_Weights

from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    confusion_matrix, f1_score, precision_recall_fscore_support
)

ImageFile.LOAD_TRUNCATED_IMAGES = False
SEED = 2026
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__, "TorchVision:", __import__("torchvision").__version__)
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory GiB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))
else:
    print("WARNING: No GPU found. Full training will likely be very slow.")

## Cell 2 — Central SMOKE/FULL configuration

On first upload **leave `RUN_MODE = "SMOKE"`**. After successfully reaching `SMOKE PASSED` on Kaggle, change **only** `RUN_MODE = "FULL"` and select **Save Version → Save & Run All**. All other fields are reproducible starting settings. SMOKE and FULL use **separate output folders/ZIP names**.

In [ ]:
from dataclasses import dataclass, asdict

# <<< CHANGE ONLY THIS LINE AFTER THE SMOKE RUN PASSES >>>
RUN_MODE = "SMOKE"  # Options: "SMOKE" or "FULL"
if RUN_MODE not in {"SMOKE", "FULL"}:
    raise ValueError("RUN_MODE must be exactly SMOKE or FULL")

@dataclass(frozen=True)
class Config:
    dataset_slug: str = "facial-dermatology-image-dataset"
    output_name: str = "efficientnet_b0_real_progressive_v1"
    model_name: str = "efficientnet_b0"
    imagenet_weights: str = "IMAGENET1K_V1"
    image_size: int = 224
    batch_size: int = 24
    num_workers: int = 2
    epochs_head: int = 4
    epochs_final_stage: int = 8
    epochs_final_two_stages: int = 8
    full_fine_tune: bool = False
    epochs_full: int = 5
    patience: int = 4
    min_epochs_per_stage: int = 2
    use_amp: bool = True
    weight_decay: float = 1e-4
    grad_clip_norm: float = 1.0
    use_class_weights: bool = True
    verify_full_sha256: bool = True
    max_suspicious_images_report: int = 100
    seed: int = SEED
    # SMOKE mode only. Sample counts are capped by availability in each class.
    smoke_train_per_class: int = 6
    smoke_val_per_class: int = 3
    smoke_test_per_class: int = 2
    smoke_batch_size: int = 8
    smoke_epochs_per_stage: int = 1
    smoke_test_optional_full_stage: bool = True

CFG = Config()
SMOKE = (RUN_MODE == "SMOKE")
CLASSES = ["Acne", "Fungal_infection", "Hyper_Pigmentation", "Normal_Skin", "Skin_cancer", "Vitiligo"]
CLASS_TO_IDX = {name: i for i, name in enumerate(CLASSES)}
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tif", ".tiff"}
OUTPUT_ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
RUN_OUTPUT_NAME = CFG.output_name + ("_SMOKE" if SMOKE else "_FULL")
OUT = OUTPUT_ROOT / RUN_OUTPUT_NAME
for folder in (OUT, OUT / "checkpoints", OUT / "plots", OUT / "reports"):
    folder.mkdir(parents=True, exist_ok=True)
(OUT / "config.json").write_text(json.dumps({**asdict(CFG), "classes": CLASSES, "class_to_idx": CLASS_TO_IDX, "run_mode": RUN_MODE,
                                                "effective_batch_size": CFG.smoke_batch_size if SMOKE else CFG.batch_size}, indent=2), encoding="utf-8")
(OUT / "class_to_idx.json").write_text(json.dumps(CLASS_TO_IDX, indent=2), encoding="utf-8")

LOGGER = logging.getLogger("dermalens_efficientnet_b0")
LOGGER.setLevel(logging.INFO)
LOGGER.propagate = False
for old_handler in list(LOGGER.handlers):
    LOGGER.removeHandler(old_handler)
fmt = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
file_handler = logging.FileHandler(OUT / "training.log", encoding="utf-8")
file_handler.setFormatter(fmt)
stream_handler = logging.StreamHandler(sys.stdout)
stream_handler.setFormatter(fmt)
LOGGER.addHandler(file_handler)
LOGGER.addHandler(stream_handler)

def save_json(path, obj):
    Path(path).write_text(json.dumps(obj, indent=2, default=str, allow_nan=False), encoding="utf-8")

LOGGER.info("Configuration saved: %s", OUT / "config.json")
LOGGER.info("RUN MODE: %s (SMOKE results are NOT scientifically valid)", RUN_MODE)
LOGGER.info("Full fine-tuning enabled in FULL mode: %s", CFG.full_fine_tune)

## Cell 3 — Detect the finalized Kaggle dataset (flat **or** nested)

Kaggle has mounted this dataset in two ways in prior runs. This cell searches both the direct `facial-dermatology-image-dataset` directory and the nested `datasets/shanmukhadattaboda/facial-dermatology-image-dataset` directory. It requires **exactly one** tree containing all three original splits. **Never** fall back to the unsplit `facial-skin-conditions-real-image-collection`.

In [ ]:
def resolve_dataset_root() -> Path:
    roots = [
        Path("/kaggle/input") / CFG.dataset_slug,
        Path("/kaggle/input/datasets/shanmukhadattaboda") / CFG.dataset_slug,
    ]
    candidates = set()
    for root in roots:
        if not root.is_dir():
            continue
        if all((root / s).is_dir() for s in ("train", "validation", "test")):
            candidates.add(root.resolve())
        # Allow one extra containing directory in the attached dataset.
        for train_dir in root.rglob("train"):
            if train_dir.is_dir() and all(
                (train_dir.parent / s).is_dir() for s in ("train", "validation", "test")
            ):
                candidates.add(train_dir.parent.resolve())
    if len(candidates) != 1:
        mounted = sorted(str(p) for p in Path("/kaggle/input").glob("**/facial-*")) if Path("/kaggle/input").is_dir() else []
        raise FileNotFoundError(
            "Attach the finalized split Kaggle dataset: "
            "shanmukhadattaboda/facial-dermatology-image-dataset. "
            "Do NOT attach only the older unsplit facial-skin-conditions-real-image-collection. "
            f"Found {len(candidates)} candidate split roots: {sorted(map(str, candidates))}; "
            f"visible facial-* mounts: {mounted}"
        )
    return next(iter(candidates))

DATA_ROOT = resolve_dataset_root()
SPLIT_DIRS = {split: DATA_ROOT / split for split in ("train", "validation", "test")}
LOGGER.info("Real dataset root: %s", DATA_ROOT)
print("Found split folders:", {key: str(path) for key, path in SPLIT_DIRS.items()})

## Cell 4 — Input audit and immutable split handling

Verify six class subfolders per split, load/decode images, record counts, flags for suspicious images, SHA-256 hashes and cross-split exact-duplicate warnings. SMOKE checks a fixed balanced subset; FULL checks **all files**. Neither mode changes or resplits the dataset. Patient-level overlap/near duplicates and clinical label quality still need independent review.

In [ ]:
def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for block in iter(lambda: fh.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

records = []
issues = []
all_split_counts = {}
random_selector = random.Random(CFG.seed)
smoke_caps = {"train": CFG.smoke_train_per_class,
              "validation": CFG.smoke_val_per_class,
              "test": CFG.smoke_test_per_class}
for split, split_root in SPLIT_DIRS.items():
    expected = set(CLASSES)
    actual = {p.name for p in split_root.iterdir() if p.is_dir()}
    if actual != expected:
        raise RuntimeError(f"Wrong class directories in {split}: missing={expected-actual}, extra={actual-expected}")
    for label in CLASSES:
        image_paths = sorted(
            p for p in (split_root / label).rglob("*")
            if p.is_file() and p.suffix.lower() in IMAGE_EXTS
        )
        if not image_paths:
            raise RuntimeError(f"No images found under {split_root / label}")
        all_split_counts[f"{split}/{label}"] = len(image_paths)
        if SMOKE:
            # Stratified fixed-seed sample, without changing any original split.
            k = min(smoke_caps[split], len(image_paths))
            if k <= 0:
                raise RuntimeError(f"SMOKE sampling found no images in {split}/{label}")
            image_paths = sorted(random_selector.sample(image_paths, k))
        for path in image_paths:
            rel = str(path.relative_to(DATA_ROOT))
            try:
                with Image.open(path) as im:
                    im.verify()
                with Image.open(path) as im:
                    im = ImageOps.exif_transpose(im).convert("RGB")
                    width, height = im.size
                    if width < 32 or height < 32:
                        issues.append({"path": rel, "issue": "very_small_image", "width": width, "height": height})
                    gray = np.asarray(im.resize((32, 32)).convert("L"), dtype=np.float32)
                    if float(gray.std()) < 3 or float(gray.mean()) < 8:
                        issues.append({"path": rel, "issue": "near_uniform_or_dark", "width": width, "height": height})
                digest = sha256_file(path) if CFG.verify_full_sha256 else "not_computed"
                records.append({
                    "split": split, "class_name": label, "class_id": CLASS_TO_IDX[label],
                    "relative_path": rel, "width": width, "height": height,
                    "file_size_bytes": path.stat().st_size, "sha256": digest
                })
            except (OSError, ValueError, UnidentifiedImageError) as exc:
                issues.append({"path": rel, "issue": "corrupt_or_unreadable", "error": str(exc)})
    LOGGER.info("Validated %s images: %d", split, sum(r["split"] == split for r in records))

manifest = pd.DataFrame.from_records(records)
manifest.to_csv(OUT / "reports" / "dataset_manifest.csv", index=False)
pd.DataFrame(issues, columns=["path", "issue", "width", "height", "error"]).to_csv(OUT / "reports" / "image_warnings.csv", index=False)

bad = [x for x in issues if x["issue"] == "corrupt_or_unreadable"]
if bad:
    raise RuntimeError(f"Found {len(bad)} corrupt images. See reports/image_warnings.csv. Fix the source dataset before training.")

if CFG.verify_full_sha256:
    overlap = (
        manifest.groupby("sha256", sort=False)
        .agg(n_splits=("split", "nunique"), count=("split", "size"))
        .reset_index()
    )
    leaked_hashes = set(overlap.loc[overlap.n_splits > 1, "sha256"])
    leakage = manifest[manifest.sha256.isin(leaked_hashes)].sort_values(["sha256", "split"])
    leakage.to_csv(OUT / "reports" / "cross_split_exact_duplicates.csv", index=False)
    if leaked_hashes:
        raise RuntimeError(f"Found {len(leaked_hashes)} exact cross-split duplicate hashes. See cross_split_exact_duplicates.csv")
    LOGGER.info("SHA-256 cross-split audit passed for %s files only.", "SAMPLED" if SMOKE else "ALL")
else:
    LOGGER.warning("Full SHA-256 disabled: exact cross-split duplicate check NOT performed.")

summary = (manifest.groupby(["split", "class_name"]).size()
           .unstack(fill_value=0).reindex(index=["train", "validation", "test"], columns=CLASSES))
summary.to_csv(OUT / "reports" / "split_class_counts.csv")
dataset_fingerprint = hashlib.sha256(
    manifest.sort_values("relative_path")[["relative_path", "sha256"]]
    .to_csv(index=False).encode("utf-8")
).hexdigest()
save_json(OUT / "reports" / "dataset_audit.json", {
    "root": str(DATA_ROOT), "fingerprint": dataset_fingerprint,
    "run_mode": RUN_MODE, "audit_scope": "sample_only" if SMOKE else "entire_dataset",
    "original_class_file_counts": all_split_counts,
    "total_images": int(len(manifest)), "split_sizes": {s: int(summary.loc[s].sum()) for s in summary.index},
    "class_names": CLASSES, "sha256_checked": CFG.verify_full_sha256,
    "exact_split_overlap_count": 0 if CFG.verify_full_sha256 else None,
    "image_warning_count": len(issues),
    "limitations": ("SMOKE verifies ONLY a selected subset; not a full dataset audit. " if SMOKE else "")
                   + "Exact hashes do not detect visually similar/augmented duplicates or patient overlap."
})
print(summary.to_string())
LOGGER.info("Dataset fingerprint: %s", dataset_fingerprint)
LOGGER.warning("Suspect/very dark images (review, not auto-removed): %d", len(issues))
if SMOKE:
    LOGGER.warning("SMOKE: audited %d sampled images, NOT the full dataset. Results are NOT for publication.", len(manifest))
else:
    LOGGER.info("FULL: audited %d images. Near-duplicate/patient checks need separate provenance.", len(manifest))


## Cell 5 — Conservative augmentation, data loaders and image normalization

Convert images to RGB, respect EXIF orientation and preserve the full facial field of view with center padding and bicubic resampling to 224×224. For **training only**, random horizontal flip (50%) and ±7° rotation. Validation/test are deterministic. Normalize using the official ImageNet B0 mean/std. This deliberately differs from the checkpoint's standard center crop in order to avoid removing lesions; document this choice in the paper.

In [ ]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)
PAD_COLOR = (128, 128, 128)

class AspectPad:
    def __init__(self, size: int, color=PAD_COLOR):
        self.size = size
        self.color = color
    def __call__(self, image):
        return ImageOps.pad(image, (self.size, self.size), method=Image.Resampling.BICUBIC, color=self.color, centering=(0.5, 0.5))

train_transform = transforms.Compose([
    AspectPad(CFG.image_size),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=7, interpolation=InterpolationMode.BILINEAR, fill=PAD_COLOR),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
valid_transform = transforms.Compose([
    AspectPad(CFG.image_size),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class FixedImageDataset(Dataset):
    def __init__(self, items, root: Path, transform):
        self.items = items.reset_index(drop=True)
        self.root = root
        self.transform = transform
    def __len__(self):
        return len(self.items)
    def __getitem__(self, index):
        row = self.items.iloc[index]
        image_path = self.root / row["relative_path"]
        with Image.open(image_path) as image:
            image = ImageOps.exif_transpose(image).convert("RGB")
            tensor = self.transform(image)
        return tensor, int(row["class_id"]), str(row["relative_path"])

train_ds = FixedImageDataset(manifest[manifest.split == "train"], DATA_ROOT, train_transform)
val_ds = FixedImageDataset(manifest[manifest.split == "validation"], DATA_ROOT, valid_transform)
test_ds = FixedImageDataset(manifest[manifest.split == "test"], DATA_ROOT, valid_transform)

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)

pin_memory = DEVICE.type == "cuda"

def make_loader(ds, *, shuffle=False):
    generator = torch.Generator().manual_seed(CFG.seed)
    return DataLoader(
        ds, batch_size=CFG.smoke_batch_size if SMOKE else CFG.batch_size, shuffle=shuffle, num_workers=CFG.num_workers,
        pin_memory=pin_memory, persistent_workers=(CFG.num_workers > 0),
        worker_init_fn=seed_worker, generator=generator
    )

train_loader = make_loader(train_ds, shuffle=True)
val_loader = make_loader(val_ds)
test_loader = make_loader(test_ds)
assert len(train_ds) and len(val_ds) and len(test_ds)
xs, ys, paths = next(iter(train_loader))
assert xs.ndim == 4 and xs.shape[1:] == (3, CFG.image_size, CFG.image_size)
assert ys.min() >= 0 and ys.max() < len(CLASSES)
LOGGER.info("Data loading sanity check passed: %s", tuple(xs.shape))

## Cell 6 — Load **authentic** ImageNet-pretrained EfficientNet-B0

Use `torchvision.models.efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)`, then replace `model.classifier[1]` with a 6-unit linear layer. If official weight loading fails, the notebook stops with a clear error — **never silently substitutes random weights**. Validate that the expected `features[0..8]` and final classifier are present.

Reference: [official PyTorch EfficientNet-B0 model and weights documentation](https://docs.pytorch.org/vision/main/models/generated/torchvision.models.efficientnet_b0.html).

In [ ]:
try:
    model = models.efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
except Exception as exc:
    raise RuntimeError(
        "Could not load official ImageNet EfficientNet-B0 pretrained weights. "
        "Enable Kaggle Internet on the first run or provide the authentic cached weights. "
        f"Original error: {exc}"
    ) from exc

if not isinstance(model.features, nn.Sequential) or len(model.features) != 9:
    raise RuntimeError("Unexpected EfficientNet-B0 features layout: expected 9 indexed groups (0..8)")
if not isinstance(model.classifier, nn.Sequential) or not isinstance(model.classifier[1], nn.Linear):
    raise RuntimeError("Unexpected EfficientNet-B0 classifier layout")
in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, len(CLASSES))
nn.init.normal_(model.classifier[1].weight, std=0.01)
nn.init.zeros_(model.classifier[1].bias)
model = model.to(DEVICE)
parameters = sum(p.numel() for p in model.parameters())
if not 3_000_000 < parameters < 7_000_000:
    raise RuntimeError(f"Unexpected parameter count {parameters}; model might not be EfficientNet-B0")
LOGGER.info("EfficientNet-B0: %d params, official %s weights, six-class head", parameters, CFG.imagenet_weights)

## Cell 7 — Progressive EfficientNet-B0 stages and learning rates

EfficientNet-B0 is organized into `features.0` (stem), `features.1`–`features.7` (MBConv stages), `features.8` (final 1×1 convolution), and `classifier` (dropout + linear head). Stage 1 learns only the head; stage 2 opens `features.7,8`; stage 3 opens `features.6,7,8`. SMOKE also checks the optional complete-backbone branch; FULL leaves it disabled by default. Use separate, lower learning rates for backbone groups and freeze BatchNorm running statistics during training for stability.

In [ ]:
train_counts = np.array([int((manifest.split.eq("train") & manifest.class_name.eq(c)).sum()) for c in CLASSES], dtype=np.float64)
if np.any(train_counts <= 0):
    raise RuntimeError("Every class must have training examples")
raw_weights = np.sqrt(train_counts.sum() / (len(CLASSES) * train_counts))
raw_weights = np.clip(raw_weights, 0.5, 2.0)
raw_weights /= raw_weights.mean()
class_weights = torch.tensor(raw_weights, dtype=torch.float32, device=DEVICE)
criterion = nn.CrossEntropyLoss(weight=class_weights if CFG.use_class_weights else None)
LOGGER.info("Train counts: %s; loss weights: %s", train_counts.astype(int).tolist(), raw_weights.round(4).tolist())

STAGE_SPECS = [
    {"name": "head", "epochs": CFG.epochs_head, "lrs": {"head": 1e-3}},
    {"name": "last_stage", "epochs": CFG.epochs_final_stage,
     "lrs": {"stage7": 2e-5, "final_conv": 3e-5, "head": 1e-4}},
    {"name": "last_two_stages", "epochs": CFG.epochs_final_two_stages,
     "lrs": {"stage6": 7e-6, "stage7": 1.5e-5, "final_conv": 2e-5, "head": 5e-5}},
]
if CFG.full_fine_tune or (SMOKE and CFG.smoke_test_optional_full_stage):
    STAGE_SPECS.append({
        "name": "full", "epochs": CFG.epochs_full,
        "lrs": {"stem": 5e-7, "stage1": 7e-7, "stage2": 1e-6,
                "stage3": 1.5e-6, "stage4": 2e-6, "stage5": 3e-6,
                "stage6": 4e-6, "stage7": 8e-6, "final_conv": 1e-5, "head": 3e-5},
    })
if SMOKE:
    STAGE_SPECS = [{**stage, "epochs": CFG.smoke_epochs_per_stage} for stage in STAGE_SPECS]
LOGGER.info("Effective %s stage schedule: %s", RUN_MODE,
            [(s["name"], s["epochs"]) for s in STAGE_SPECS])

def group_for_parameter(name: str) -> str:
    if name.startswith("classifier."):
        return "head"
    if name.startswith("features.0."):
        return "stem"
    if name.startswith("features.8."):
        return "final_conv"
    for i in range(1, 8):
        if name.startswith(f"features.{i}."):
            return f"stage{i}"
    raise RuntimeError(f"Unrecognized EfficientNet-B0 parameter: {name}")

def prepare_stage(model, stage):
    for p in model.parameters():
        p.requires_grad_(False)
    buckets = defaultdict(list)
    for name, p in model.named_parameters():
        group = group_for_parameter(name)
        if group in stage["lrs"]:
            p.requires_grad_(True)
            buckets[group].append(p)
    if set(buckets) != set(stage["lrs"]):
        raise RuntimeError(f"Missing optimizer groups: {set(stage['lrs']) - set(buckets)}")
    optimizer = torch.optim.AdamW(
        [{"params": buckets[group], "lr": lr, "group_name": group}
         for group, lr in stage["lrs"].items()],
        weight_decay=CFG.weight_decay
    )
    count = sum(p.numel() for p in model.parameters() if p.requires_grad)
    LOGGER.info("Stage %s: trainable parameters = %d; LR groups = %s", stage["name"], count, stage["lrs"])
    return optimizer

def freeze_bn_running_stats(module):
    if isinstance(module, nn.modules.batchnorm._BatchNorm):
        module.eval()

def metric_dict(y_true, y_pred, loss):
    return {
        "loss": float(loss),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, labels=list(range(len(CLASSES))), average="macro", zero_division=0)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
    }

model.eval()
with torch.inference_mode():
    logits = model(xs[:min(2, len(xs))].to(DEVICE))
assert logits.shape == (min(2, len(xs)), len(CLASSES))
del xs, ys, paths, logits
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()
LOGGER.info("EfficientNet-B0 model/label contract passed.")

## Cell 8 — Train/eval functions, AMP and atomic checkpointing

Every training epoch logs loss, macro-F1, balanced accuracy, stage name, learning rates and duration. Use AdamW, gradient clipping and AMP where available. Validation/test predictions retain **six class probabilities** for downstream real/synthetic comparison. Write the full model `state_dict` (with class mapping and preprocessing metadata) to a temporary file and atomically rename it.

In [ ]:
AMP_ENABLED = CFG.use_amp and DEVICE.type == "cuda"
SCALER = torch.amp.GradScaler("cuda", enabled=AMP_ENABLED)
HISTORY_PATH = OUT / "reports" / "history.csv"
history_rows = []

def amp_context():
    return torch.amp.autocast(device_type="cuda", enabled=True) if AMP_ENABLED else nullcontext()

def run_epoch(model, loader, *, optimizer=None):
    is_train = optimizer is not None
    model.train(is_train)
    if is_train:
        model.apply(freeze_bn_running_stats)
    running_loss, seen = 0.0, 0
    truths, guesses, prediction_rows = [], [], []
    context = torch.enable_grad() if is_train else torch.inference_mode()
    with context:
        for batch_x, batch_y, batch_paths in loader:
            batch_x = batch_x.to(DEVICE, non_blocking=pin_memory)
            batch_y = batch_y.to(DEVICE, non_blocking=pin_memory)
            if is_train:
                optimizer.zero_grad(set_to_none=True)
            with amp_context():
                batch_logits = model(batch_x)
                loss = criterion(batch_logits, batch_y)
            if not torch.isfinite(loss):
                raise FloatingPointError("Non-finite loss detected; stopped to protect checkpoints")
            if is_train:
                SCALER.scale(loss).backward()
                SCALER.unscale_(optimizer)
                nn.utils.clip_grad_norm_((p for p in model.parameters() if p.requires_grad), CFG.grad_clip_norm)
                SCALER.step(optimizer)
                SCALER.update()
            probs = torch.softmax(batch_logits.detach().float(), dim=1)
            predictions = probs.argmax(dim=1)
            y = batch_y.detach().cpu().tolist()
            yhat = predictions.detach().cpu().tolist()
            truths.extend(y)
            guesses.extend(yhat)
            if not is_train:
                all_probs = probs.detach().cpu().numpy()
                for row_path, y_true, y_pred, probability_vector in zip(batch_paths, y, yhat, all_probs):
                    row = {"relative_path": row_path, "true_id": int(y_true), "pred_id": int(y_pred),
                           "true_label": CLASSES[y_true], "pred_label": CLASSES[y_pred],
                           "confidence": float(probability_vector[y_pred])}
                    row.update({f"prob_{c}": float(probability_vector[i]) for i, c in enumerate(CLASSES)})
                    prediction_rows.append(row)
            batch_size = len(batch_y)
            running_loss += float(loss.detach().item()) * batch_size
            seen += batch_size
    assert seen > 0
    metrics = metric_dict(truths, guesses, running_loss / seen)
    return metrics, prediction_rows

def save_checkpoint(file_path, model, stage, epoch, val_metrics):
    payload = {
        "model_name": CFG.model_name, "run_mode": RUN_MODE,
        "checkpoint_purpose": "pipeline_verification_only" if SMOKE else "research_training",
        "weights_origin": CFG.imagenet_weights,
        "architecture": "torchvision.models.efficientnet_b0",
        "model_state_dict": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
        "num_classes": len(CLASSES), "classes": CLASSES, "class_to_idx": CLASS_TO_IDX,
        "preprocessing": {"input_size": CFG.image_size, "resize": "aspect_ratio_pad", "interpolation": "bicubic",
                          "pad_rgb": list(PAD_COLOR), "mean": list(IMAGENET_MEAN),
                          "std": list(IMAGENET_STD), "exif_transpose": True},
        "stage": stage, "stage_epoch": int(epoch), "validation": val_metrics,
        "dataset_fingerprint": dataset_fingerprint,
        "random_seed": CFG.seed,
        "torch_version": str(torch.__version__),
        "created_utc": datetime.now(timezone.utc).isoformat()
    }
    temp = Path(str(file_path) + ".tmp")
    torch.save(payload, temp)
    os.replace(temp, file_path)

# Compare validation macro-F1 first; on exact ties, prefer lower validation loss.
def better(candidate, incumbent, eps=1e-8):
    if incumbent is None:
        return True
    if candidate["macro_f1"] > incumbent["macro_f1"] + eps:
        return True
    return (abs(candidate["macro_f1"] - incumbent["macro_f1"]) <= eps
            and candidate["loss"] < incumbent["loss"] - eps)

## Cell 9 — Fully automated progressive fine-tuning

Train stage by stage; save epoch history and stage-best checkpoints; use cosine decay and early stopping. Each next stage begins from the preceding stage's best validation checkpoint. Select global best **only by validation macro-F1**, with validation loss tie-break. The SMOKE code path is identical except smaller samples and one epoch/stage; exceptions write `error_traceback.txt` and `run_status.json`.

In [ ]:
def safe_load_checkpoint(file_path, map_location="cpu"):
    try:
        return torch.load(file_path, map_location=map_location, weights_only=True)
    except TypeError as exc:
        raise RuntimeError("This notebook requires PyTorch with torch.load(weights_only=True).") from exc

def run_all_stages():
    global_best = None
    global_path = OUT / "checkpoints" / "best_model.pt"
    stage_results = []
    for stage in STAGE_SPECS:
        name = stage["name"]
        LOGGER.info("=== BEGIN STAGE: %s | max epochs: %d ===", name, stage["epochs"])
        optimizer = prepare_stage(model, stage)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, stage["epochs"]), eta_min=0)
        stage_best = None
        stage_path = OUT / "checkpoints" / f"best_{name}.pt"
        stagnant = 0
        for epoch in range(1, stage["epochs"] + 1):
            tick = time.perf_counter()
            train_metrics, _ = run_epoch(model, train_loader, optimizer=optimizer)
            validation_metrics, _ = run_epoch(model, val_loader)
            row = {"stage": name, "epoch_in_stage": epoch,
                   "train_loss": train_metrics["loss"], "train_accuracy": train_metrics["accuracy"],
                   "train_macro_f1": train_metrics["macro_f1"],
                   "val_loss": validation_metrics["loss"], "val_accuracy": validation_metrics["accuracy"],
                   "val_macro_f1": validation_metrics["macro_f1"],
                   "val_balanced_accuracy": validation_metrics["balanced_accuracy"],
                   "seconds": round(time.perf_counter() - tick, 2)}
            row.update({f"lr_{group['group_name']}": float(group["lr"]) for group in optimizer.param_groups})
            history_rows.append(row)
            pd.DataFrame(history_rows).to_csv(HISTORY_PATH, index=False)
            LOGGER.info("%s epoch %02d/%02d | train_loss=%.4f val_loss=%.4f val_macro_f1=%.4f val_acc=%.4f sec=%.1f",
                        name, epoch, stage["epochs"], train_metrics["loss"], validation_metrics["loss"],
                        validation_metrics["macro_f1"], validation_metrics["accuracy"], row["seconds"])
            if better(validation_metrics, stage_best):
                stage_best = dict(validation_metrics)
                save_checkpoint(stage_path, model, name, epoch, validation_metrics)
                stagnant = 0
            else:
                stagnant += 1
            if better(validation_metrics, global_best):
                global_best = dict(validation_metrics)
                save_checkpoint(global_path, model, name, epoch, validation_metrics)
                LOGGER.info("Updated global best at %s epoch %d: val_macro_f1=%.4f", name, epoch, validation_metrics["macro_f1"])
            scheduler.step()
            if epoch >= CFG.min_epochs_per_stage and stagnant >= CFG.patience:
                LOGGER.info("Early stopping %s after epoch %d", name, epoch)
                break
        if stage_best is None:
            raise RuntimeError(f"Stage {name} completed without a checkpoint")
        stage_results.append({"stage": name, "best_val": stage_best, "checkpoint": str(stage_path), "completed_epochs": epoch})
        save_json(OUT / "reports" / "stage_results.json", stage_results)
        model.load_state_dict(safe_load_checkpoint(stage_path, DEVICE)["model_state_dict"], strict=True)
        LOGGER.info("Reloaded best checkpoint from stage %s before next stage.", name)
    if not global_path.is_file():
        raise RuntimeError("No globally selected model checkpoint found")
    return global_path, stage_results

save_json(OUT / "run_status.json", {"status": "smoke_running" if SMOKE else "full_running", "run_mode": RUN_MODE, "started_utc": datetime.now(timezone.utc).isoformat()})
try:
    BEST_PATH, STAGE_RESULTS = run_all_stages()
    save_json(OUT / "run_status.json", {"status": "smoke_trained_not_yet_evaluated" if SMOKE else "full_trained_not_yet_evaluated", "run_mode": RUN_MODE, "best_checkpoint": str(BEST_PATH),
                                        "updated_utc": datetime.now(timezone.utc).isoformat()})
except Exception as exc:
    tb = traceback.format_exc()
    (OUT / "error_traceback.txt").write_text(tb, encoding="utf-8")
    save_json(OUT / "run_status.json", {"status": "failed_during_training", "run_mode": RUN_MODE, "error": str(exc)})
    LOGGER.exception("Training failed")
    raise

## Cell 10 — Final independent checkpoint reload and evaluation

After model selection, reload the winning checkpoint and generate validation and test predictions (six class probabilities each). Evaluate the final real test only **after** validation-based model selection. SMOKE test is a software diagnostic, not reportable research data. FULL results still need data-quality and protocol verification.

In [ ]:
best_payload = safe_load_checkpoint(BEST_PATH, DEVICE)
assert best_payload["class_to_idx"] == CLASS_TO_IDX
assert best_payload["dataset_fingerprint"] == dataset_fingerprint
model.load_state_dict(best_payload["model_state_dict"], strict=True)
model.to(DEVICE).eval()

selected_val_metrics, selected_val_rows = run_epoch(model, val_loader)
val_predictions = pd.DataFrame(selected_val_rows)
val_predictions.to_csv(OUT / "reports" / "validation_predictions.csv", index=False)

if SMOKE:
    LOGGER.warning("SMOKE TEST metrics are pipeline diagnostics, NOT held-out research results!")
else:
    LOGGER.info("FULL: held-out test evaluation begins; do not change training settings from test feedback.")
test_metrics, test_rows = run_epoch(model, test_loader)
test_predictions = pd.DataFrame(test_rows)
test_predictions.to_csv(OUT / "reports" / "test_predictions.csv", index=False)

truths = test_predictions.true_id.astype(int).tolist()
preds = test_predictions.pred_id.astype(int).tolist()
report_dict = classification_report(truths, preds, labels=list(range(len(CLASSES))),
                                    target_names=CLASSES, zero_division=0, output_dict=True)
report_text = classification_report(truths, preds, labels=list(range(len(CLASSES))),
                                    target_names=CLASSES, zero_division=0)
save_json(OUT / "reports" / "test_classification_report.json", report_dict)
(OUT / "reports" / "test_classification_report.txt").write_text(report_text, encoding="utf-8")
cm = confusion_matrix(truths, preds, labels=list(range(len(CLASSES))))
pd.DataFrame(cm, index=CLASSES, columns=CLASSES).to_csv(OUT / "reports" / "test_confusion_matrix.csv")

metrics_output = {
    "model": CFG.model_name, "training_data": "real_only",
    "run_mode": RUN_MODE, "evaluation_scope": "smoke_subset_debug_only" if SMOKE else "full_holdout_test",
    "publishable": not SMOKE,
    "selection_metric": "validation_macro_f1",
    "selected_stage": best_payload["stage"], "selected_epoch_within_stage": best_payload["stage_epoch"],
    "best_validation_at_selection": best_payload["validation"],
    "validation_after_reload": selected_val_metrics,
    "test": test_metrics,
    "test_classification_report": report_dict,
    "dataset_fingerprint": dataset_fingerprint,
    "test_samples": len(truths), "classes": CLASSES,
    "warning": ("SMOKE SUBSET METRICS ARE INVALID FOR SCIENTIFIC REPORTING. " if SMOKE else "")
               + "Research-only image classification. Test set must not be used for model selection."
}
save_json(OUT / "reports" / "final_metrics.json", metrics_output)
LOGGER.info("SELECTED: %s epoch %d", best_payload["stage"], best_payload["stage_epoch"])
LOGGER.info("FINAL TEST: %s", test_metrics)
print("\n=== " + ("SMOKE / NON-REPORTABLE" if SMOKE else "FULL HOLDOUT") + " TEST REPORT ===\n", report_text)

## Cell 11 — Figures

Write training history, raw confusion matrix, and normalized confusion matrix PNGs. These can later be compared with the five larger model families, but do not interpret smoke-run charts scientifically.

In [ ]:
hist = pd.read_csv(HISTORY_PATH)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for stage_name, rows in hist.groupby("stage", sort=False):
    step = rows.index.to_numpy() + 1
    axes[0].plot(step, rows.train_loss, marker=".", label=f"{stage_name} train")
    axes[0].plot(step, rows.val_loss, marker=".", linestyle="--", label=f"{stage_name} val")
    axes[1].plot(step, rows.val_macro_f1, marker="o", label=stage_name)
axes[0].set(title="Training and validation loss", xlabel="Global epoch", ylabel="Cross-entropy loss")
axes[1].set(title="Validation macro-F1", xlabel="Global epoch", ylabel="Macro-F1", ylim=(0, 1.02))
for ax in axes:
    ax.grid(alpha=0.2)
    ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(OUT / "plots" / "learning_curves.png", dpi=160)
plt.close(fig)

for normalized in (False, True):
    plot_cm = cm.astype(float)
    if normalized:
        denom = plot_cm.sum(axis=1, keepdims=True)
        plot_cm = np.divide(plot_cm, denom, out=np.zeros_like(plot_cm), where=denom > 0)
    fig, ax = plt.subplots(figsize=(10, 8))
    im = ax.imshow(plot_cm, cmap="Blues", vmin=0, vmax=1 if normalized else None)
    ax.set_xticks(range(len(CLASSES)), CLASSES, rotation=35, ha="right")
    ax.set_yticks(range(len(CLASSES)), CLASSES)
    ax.set_xlabel("Predicted class")
    ax.set_ylabel("True class")
    ax.set_title("Held-out test confusion matrix" + (" (row-normalized)" if normalized else " (counts)"))
    for i in range(len(CLASSES)):
        for j in range(len(CLASSES)):
            text_value = f"{plot_cm[i, j]:.2f}" if normalized else str(int(plot_cm[i, j]))
            ax.text(j, i, text_value, ha="center", va="center", fontsize=9,
                    color="white" if plot_cm[i, j] > (0.5 if normalized else cm.max()/2) else "black")
    fig.colorbar(im, ax=ax, fraction=0.046)
    fig.tight_layout()
    fig.savefig(OUT / "plots" / ("confusion_normalized.png" if normalized else "confusion_counts.png"), dpi=160)
    plt.close(fig)
LOGGER.info("Saved learning curves and confusion matrices.")

## Cell 12 — Portable evaluation loader and independent inference test

Export a standalone `inference_example.py` that reconstructs EfficientNet-B0 with **`weights=None`** (no Internet needed for inference), reloads the six-class checkpoint, and applies the **same** RGB, EXIF, aspect-preserving bicubic-padding and ImageNet normalization. Test the script independently against the notebook's validation predictions. The saved checkpoint works for later real and independently held-out synthetic evaluation.

In [ ]:
PREPROCESS_META = best_payload["preprocessing"]
save_json(OUT / "preprocessing.json", PREPROCESS_META)
inference_script = r'''"""Reusable DermaLens EfficientNet-B0 inference (research only).

Usage:
    python inference_example.py --checkpoint /path/to/best_model.pt --image /path/to/face.jpg

Offline when checkpoint is present: torchvision builds architecture with weights=None.
"""
import argparse
import json
from pathlib import Path

import torch
from torch import nn
from torchvision import models, transforms
from PIL import Image, ImageOps


def load_classifier(checkpoint_path, device="cpu"):
    payload = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
    labels = payload["classes"]
    model = models.efficientnet_b0(weights=None)
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, len(labels))
    model.load_state_dict(payload["model_state_dict"], strict=True)
    model.to(device).eval()
    return model, payload


def make_preprocess(meta):
    size = int(meta["input_size"])
    color = tuple(meta["pad_rgb"])
    def pad(image):
        return ImageOps.pad(image, (size, size), method=Image.Resampling.BICUBIC, color=color)
    return transforms.Compose([
        pad,
        transforms.ToTensor(),
        transforms.Normalize(meta["mean"], meta["std"])
    ])


@torch.inference_mode()
def predict(checkpoint_path, image_path, device=None):
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    model, payload = load_classifier(checkpoint_path, device=device)
    transform = make_preprocess(payload["preprocessing"])
    with Image.open(image_path) as im:
        img = ImageOps.exif_transpose(im).convert("RGB")
        x = transform(img).unsqueeze(0).to(device)
    probs = torch.softmax(model(x), dim=1)[0].cpu().tolist()
    pred_id = max(range(len(probs)), key=probs.__getitem__)
    return {"prediction": payload["classes"][pred_id],
            "probabilities": {name: float(p) for name, p in zip(payload["classes"], probs)},
            "caution": "Research only. Not for clinical diagnosis."}

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--checkpoint", required=True)
    parser.add_argument("--image", required=True)
    args = parser.parse_args()
    print(json.dumps(predict(args.checkpoint, args.image), indent=2))
'''
(OUT / "inference_example.py").write_text(inference_script, encoding="utf-8")
compile(inference_script, str(OUT / "inference_example.py"), "exec")

reuse_metadata = {
    "artifact_schema_version": "dermalens-classifier-v1",
    "model": CFG.model_name, "run_mode": RUN_MODE,
    "publishable": not SMOKE,
    "checkpoint_relative_path": "checkpoints/best_model.pt",
    "class_map_relative_path": "class_to_idx.json",
    "preprocess_relative_path": "preprocessing.json",
    "metrics_relative_path": "reports/final_metrics.json",
    "test_predictions_relative_path": "reports/test_predictions.csv",
    "dataset_fingerprint": dataset_fingerprint,
    "architecture": "torchvision.models.efficientnet_b0(weights=None) + Linear(in_features,6)",
    "checkpoint_load": "torch.load(path, map_location='cpu', weights_only=True)",
    "test_set_locked": True,
    "note": "Use this checkpoint as inference-only input for later real/synthetic evaluation."
}
save_json(OUT / "artifact_manifest.json", reuse_metadata)
independent_ns = {"__name__": "dermalens_reuse_check"}
exec(compile(inference_script, "inference_example.py", "exec"), independent_ns)
probe_image = DATA_ROOT / str(val_predictions.iloc[0]["relative_path"])
reuse_out = independent_ns["predict"](str(BEST_PATH), str(probe_image), device="cpu")
expected = val_predictions.iloc[0]
for class_name in CLASSES:
    if abs(float(reuse_out["probabilities"][class_name]) - float(expected[f"prob_{class_name}"])) > 1e-3:
        raise AssertionError(f"Independent inference mismatch for {class_name}")
save_json(OUT / "reports" / "inference_reload_check.json", {
    "passed": True, "mode": RUN_MODE,
    "image": str(expected["relative_path"]),
    "predicted_class": reuse_out["prediction"], "probability_tolerance": 0.001,
    "note": "Independent CPU model reload and preprocessing agree with validation predictions."
})
LOGGER.info("Independent inference script and checkpoint reload verified.")

## Cell 13 — Save reproducibility environment, run status and reusable ZIP

Record Python/PyTorch/TorchVision versions, GPU, audit scope, best stage, input fingerprint and measured metrics; write `run_status.json`, an in-artifact README and a ZIP under `/kaggle/working`. The ZIP contains `checkpoints/best_model.pt`, stage checkpoints, `class_to_idx.json`, `preprocessing.json`, predictions, history, plots, logs and portable inference. Keep the saved Kaggle **version** for reproducibility.

In [ ]:
package_info = {
    "python": sys.version.split()[0],
    "torch": str(torch.__version__),
    "torchvision": __import__("torchvision").__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "pillow": Image.__version__ if hasattr(Image, "__version__") else "unknown",
    "sklearn": __import__("sklearn").__version__,
    "cuda_version": torch.version.cuda,
    "cuda_available": torch.cuda.is_available(),
    "gpu": torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else None,
    "stage_count": len(STAGE_SPECS), "run_mode": RUN_MODE,
    "audit_scope": "subset_only" if SMOKE else "full_dataset",
    "best_stage": best_payload["stage"],
    "best_stage_epoch": best_payload["stage_epoch"],
    "dataset_fingerprint": dataset_fingerprint,
}
save_json(OUT / "environment.json", package_info)

readme = f"""# DermaLens EfficientNet-B0 Real-Only Progressive Fine-Tuning

## MODE: {RUN_MODE}
- **RESEARCH STATUS:** {"SMOKE CHECK ONLY — NOT REPORTABLE" if SMOKE else "FULL REAL-DATA TRAINING"}
- To run the complete experiment, change `RUN_MODE = "SMOKE"` to `RUN_MODE = "FULL"` in the central settings cell and Save & Run All.
- SMOKE audits only sampled files and gives non-publishable debugging metrics.

## Checkpoint
- `checkpoints/best_model.pt`: selected by validation macro-F1 (ties: lower validation loss).
- `checkpoints/best_<stage>.pt`: best checkpoint from each completed stage.
- Selected stage: **{best_payload['stage']}**, epoch **{best_payload['stage_epoch']}**.
- Dataset fingerprint: `{dataset_fingerprint}`.

## Inputs and outputs
- Input: frozen `train/`, `validation/`, `test/` Kaggle real dataset.
- `config.json`: reproducible training configuration.
- `preprocessing.json`, `class_to_idx.json`: loader contract.
- `artifact_manifest.json`: downstream evaluation adapter contract.
- `reports/dataset_manifest.csv`, `reports/split_class_counts.csv`: immutable input audit.
- `reports/history.csv`, `reports/stage_results.json`: complete training history.
- `reports/final_metrics.json`, `reports/test_classification_report.*`: final evaluations.
- `reports/test_predictions.csv`: all per-image test probabilities.
- `plots/*.png`: training curves and confusion matrices.
- `training.log`: complete application log.
- `inference_example.py`: inference without re-downloading pretrained weights.

## Loading the artifact in another Kaggle notebook
1. Add this **notebook's saved output** as Input.
2. Locate `checkpoints/best_model.pt` under `/kaggle/input/`.
3. Use `inference_example.py` or reconstruct `torchvision.models.efficientnet_b0(weights=None)` with a 6-unit classifier head.
4. `torch.load(path, map_location='cpu', weights_only=True)` and use the embedded `preprocessing` metadata.

## Research precautions
- This is research software, **not a validated clinical diagnostic system**.
- The held-out test was evaluated only once for the finalized run; do not tune to the test results.
- Exact SHA-256 cross-split matching cannot detect near-duplicates or same-patient overlap.
- Default run uses one seed; additional independent seeds are needed for uncertainty estimates.
- Results cannot substantiate patient-level generalization without patient-disjoint identifiers.
"""
(OUT / "README.md").write_text(readme, encoding="utf-8")
zip_path = OUTPUT_ROOT / f"{RUN_OUTPUT_NAME}_artifact.zip"
# Finalize metadata, then archive outputs in one pass.
save_json(OUT / "run_status.json", {
    "status": "smoke_passed" if SMOKE else "full_completed", "run_mode": RUN_MODE,
    "publishable": not SMOKE, "updated_utc": datetime.now(timezone.utc).isoformat(),
    "checkpoint": str(BEST_PATH), "zip": str(zip_path),
    "selected_stage": best_payload["stage"], "val_macro_f1": float(best_payload["validation"]["macro_f1"]),
    "test_macro_f1": float(test_metrics["macro_f1"])
})
if zip_path.exists():
    zip_path.unlink()
with zipfile.ZipFile(zip_path, mode="w", compression=zipfile.ZIP_DEFLATED, compresslevel=3,
                     allowZip64=True) as bundle:
    for path in sorted(OUT.rglob("*")):
        if path.is_file() and not path.name.endswith(".tmp"):
            bundle.write(path, arcname=str(path.relative_to(OUT)))

print("\n=== " + RUN_MODE + " RUN COMPLETED ===")
print("Selected stage:", best_payload["stage"])
print("Selected validation macro-F1:", round(best_payload["validation"]["macro_f1"], 4))
print("Final test macro-F1:", round(test_metrics["macro_f1"], 4))
print("Best checkpoint:", BEST_PATH)
print("Reusable artifact ZIP:", zip_path, "size MiB:", round(zip_path.stat().st_size / 2**20, 1))
LOGGER.info("%s pipeline completed; outputs exported.", RUN_MODE)
if SMOKE:
    print("SMOKE PASSED: code paths executed on a tiny subset. NOT a full dataset audit or research result.")
    print('NEXT: Change RUN_MODE = "SMOKE" to RUN_MODE = "FULL" in Cell 2, then Save & Run All.')

## How to use the notebook in Kaggle

1. Import this `.ipynb` into Kaggle. Attach the **finalized** `shanmukhadattaboda/facial-dermatology-image-dataset` with existing train/validation/test folders.
2. Enable **GPU T4**, **Internet on first run**, and leave `RUN_MODE = "SMOKE"`. Click **Save Version → Save & Run All**.
3. Verify **`SMOKE PASSED`** and `efficientnet_b0_real_progressive_v1_SMOKE/run_status.json` = `"smoke_passed"`. Smoke tests the authentic TorchVision architecture, checkpoint reload and export on Kaggle.
4. Switch only `RUN_MODE = "FULL"`, save a **new** Kaggle version, and execute all cells. Leave `full_fine_tune = False` unless we agree to a separate controlled experiment.
5. Confirm `efficientnet_b0_real_progressive_v1_FULL/run_status.json` = `"full_completed"`. Save `efficientnet_b0_real_progressive_v1_FULL_artifact.zip` and record the Kaggle version ID.
6. For downstream real-vs-synthetic evaluation, add the **FULL** saved output as Kaggle Input; load `checkpoints/best_model.pt` with `inference_example.py` and use **the original class mapping and preprocessing**.

**Do not change any held-out test samples, resplit the dataset, tune hyperparameters using test outcomes, or treat smoke metrics as publishable.** Full T4 execution and all-image integrity remain to be established by your Kaggle run.